<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

Зоопарк

----

### Вариант задания 

Система «Зоопарк»: животные, зоопарк с коллекциями и смотритель, реагирующий на события.


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Продолжение проекта «Зоопарк». Базовый класс `Animal` (4 атрибута и 4 метода) и производные `Lion`, `Monkey`, `Penguin`. Коллекции в `Zoo`: `List<Animal>`, `Dictionary<string, Animal>`, `HashSet<string>`, `Queue<Animal>`. Делегаты: `AnimalFilter`, `Action<Animal>`. События: `Animal.BecameHungry`, `Zoo.AnimalAdded`, `Zoo.AnimalHungry`; подписчики — лямбда и метод `Zookeeper.OnHungry`.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// ===== Делегат =====
public delegate bool AnimalFilter(Animal animal);

// ===== Базовый класс: 4 атрибута, 4 метода =====
public abstract class Animal
{
    private string _name;
    private int _age;
    private double _weight;
    private int _energy = 50;

    public string Name   { get { return _name; }   set { _name = value; } }
    public int Age       { get { return _age; }    set { _age = value < 0 ? 0 : value; } }
    public double Weight { get { return _weight; } set { _weight = value; } }

    public int Energy
    {
        get { return _energy; }
        protected set
        {
            int old = _energy;
            _energy = Math.Max(0, Math.Min(100, value));
            // Событие срабатывает только при переходе в состояние «голоден»
            if (old > 20 && _energy <= 20)
                BecameHungry?.Invoke(this);
        }
    }

    // ===== Событие =====
    public event Action<Animal> BecameHungry;

    protected Animal(string name, int age, double weight)
    {
        Name = name;
        Age = age;
        Weight = weight;
    }

    public abstract void MakeSound();

    public void Eat(string food, int grams)
    {
        Energy += grams / 20;
        Weight += grams / 1000.0;
        Console.WriteLine($"{Name} съел {food} ({grams} г). Энергия: {Energy}");
    }

    public void Sleep()
    {
        Energy += 30;
        Console.WriteLine($"{Name} поспал. Энергия: {Energy}");
    }

    public virtual void DisplayInfo() =>
        Console.WriteLine($"{GetType().Name} {Name}, {Age} лет, {Weight:F1} кг, энергия {Energy}");
}

// ===== Производные классы =====
public class Lion : Animal
{
    public int PrideSize { get; set; } = 5;
    public int RoarPower { get; set; } = 80;
    public double ManeLength { get; set; } = 0.4;

    public Lion(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Рррр!");
    public void Roar() => Console.WriteLine($"{Name} рычит с силой {RoarPower}");
    public void Hunt() { Energy -= 35; Console.WriteLine($"{Name} охотится. Энергия: {Energy}"); }
    public void LeadPride() => Console.WriteLine($"{Name} ведёт прайд из {PrideSize} особей");
}

public class Monkey : Animal
{
    public int Intelligence { get; set; } = 7;
    public double TailLength { get; set; } = 0.6;
    public int Bananas { get; set; }

    public Monkey(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Уу-уу!");
    public void Climb() { Energy -= 15; Console.WriteLine($"{Name} лезет на дерево. Энергия: {Energy}"); }
    public void CollectBanana() { Bananas++; Console.WriteLine($"{Name} собрал бананов: {Bananas}"); }
    public void UseTool() => Console.WriteLine($"{Name} использует инструмент");
}

public class Penguin : Animal
{
    public double DiveDepth { get; set; } = 100;
    public int FeatherCount { get; set; } = 30000;
    public string Colony { get; set; } = "Антарктическая";

    public Penguin(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Кррря!");
    public void Slide() => Console.WriteLine($"{Name} скользит на животе");
    public void Dive() { Energy -= 20; Console.WriteLine($"{Name} ныряет на {DiveDepth} м. Энергия: {Energy}"); }
}

// ===== Зоопарк: коллекции, делегаты, события =====
public class Zoo
{
    // Коллекции
    private List<Animal> _animals = new List<Animal>();                 // порядок поступления
    private Dictionary<string, Animal> _byName = new Dictionary<string, Animal>(); // быстрый поиск
    private HashSet<string> _species = new HashSet<string>();           // уникальные виды
    private Queue<Animal> _feedQueue = new Queue<Animal>();             // очередь на кормление

    public string Title { get; set; }
    public int Count => _animals.Count;
    public int SpeciesCount => _species.Count;

    // События зоопарка
    public event Action<Animal> AnimalAdded;
    public event Action<Animal> AnimalHungry;

    public Zoo(string title)
    {
        Title = title;
    }

    public bool Add(Animal animal)
    {
        if (!_byName.TryAdd(animal.Name, animal))
        {
            Console.WriteLine($"Животное с именем {animal.Name} уже есть в зоопарке");
            return false;
        }
        _animals.Add(animal);
        _species.Add(animal.GetType().Name);

        // Зоопарк подписывается на событие животного
        animal.BecameHungry += OnAnimalHungry;

        AnimalAdded?.Invoke(animal);
        return true;
    }

    private void OnAnimalHungry(Animal animal)
    {
        _feedQueue.Enqueue(animal);
        AnimalHungry?.Invoke(animal);
    }

    public Animal Get(string name)
    {
        return _byName.TryGetValue(name, out Animal a) ? a : null;
    }

    // Фильтрация через делегат
    public List<Animal> Find(AnimalFilter filter)
    {
        List<Animal> result = new List<Animal>();
        foreach (Animal a in _animals)
            if (filter(a)) result.Add(a);
        return result;
    }

    // Применение действия ко всем животным (Action как параметр)
    public void ForEach(Action<Animal> action)
    {
        foreach (Animal a in _animals) action(a);
    }

    public int HungryInQueue => _feedQueue.Count;

    public Animal NextToFeed()
    {
        return _feedQueue.Count > 0 ? _feedQueue.Dequeue() : null;
    }
}

public class Zookeeper
{
    public string Name { get; set; }
    public Zookeeper(string name) { Name = name; }

    // Обработчик события
    public void OnHungry(Animal animal)
    {
        Console.WriteLine($"Смотритель {Name}: {animal.Name} голоден!");
    }
}

// ===== Демонстрация =====
Zoo zoo = new Zoo("Городской зоопарк");
Zookeeper keeper = new Zookeeper("Иван");

// Подписка на события (метод и лямбда)
zoo.AnimalAdded += a => Console.WriteLine($"В зоопарк поступил: {a.GetType().Name} {a.Name}");
zoo.AnimalHungry += keeper.OnHungry;

Lion lion = new Lion("Симба", 6, 190);
Monkey monkey = new Monkey("Чита", 4, 25);
Penguin penguin = new Penguin("Пингви", 3, 15);

zoo.Add(lion);
zoo.Add(monkey);
zoo.Add(penguin);
zoo.Add(new Lion("Симба", 2, 100));   // дубликат имени

Console.WriteLine($"Животных: {zoo.Count}, видов: {zoo.SpeciesCount}");

// Делегат-фильтр и лямбды
List<Animal> heavy = zoo.Find(a => a.Weight > 20);
Console.WriteLine("Тяжелее 20 кг: " + string.Join(", ", heavy.Select(a => a.Name)));

zoo.ForEach(a => a.MakeSound());

// Срабатывание события BecameHungry
lion.Hunt();
lion.Hunt();        // энергия падает ниже порога -> событие
monkey.Climb();
monkey.Climb();
monkey.Climb();     // тоже проголодается

// Очередь на кормление
Console.WriteLine($"В очереди на кормление: {zoo.HungryInQueue}");
Animal next;
while ((next = zoo.NextToFeed()) != null)
{
    next.Eat("корм", 1000);
}

zoo.Get("Чита")?.DisplayInfo();